# 18. Recursion in practice

After this notebook you'll be able to:

- follow a function that calls itself **twice**, like Fibonacci,
- put a **helper** function inside a LeetCode method, the way most solutions do,
- change an outer variable from a helper with `nonlocal`,
- choose between two styles: return the answer up, or collect into a list,
- make slow recursion fast with `@cache`.

You know the basics from the last notebook: a base case, a smaller step, and the caller waits for the call to finish. This notebook is about using that in real problems.

---

## Two calls at once: Fibonacci

The **Fibonacci** numbers start 0, 1, and then each one is the sum of the two before it:

| `n` | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| `fib(n)` | 0 | 1 | 1 | 2 | 3 | 5 | 8 | 13 |

So `fib(n)` is `fib(n - 1) + fib(n - 2)`. The first two, `fib(0) = 0` and `fib(1) = 1`, are the base case.

In [ ]:
def fib(n):
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

print(fib(1))
print(fib(2))
print(fib(10))

```
1
1
55
```

`if n < 2: return n` covers both base cases at once: `fib(0)` returns 0 and `fib(1)` returns 1.

### The call tree

This function calls itself **twice**. Each of those calls can call twice again. Drawn out, `fib(4)` looks like a tree, upside down:

```
                    fib(4)
                 /          \
            fib(3)            fib(2)
           /      \          /      \
       fib(2)    fib(1)   fib(1)    fib(0)
      /      \
  fib(1)    fib(0)
```

Python works through it **left first**: it goes all the way down the left side, comes back, then does the right side. Here it is with prints, pushed right by depth like before:

In [ ]:
def fib_traced(n, depth):
    print("    " * depth + f"fib({n})")
    if n < 2:
        return n
    return fib_traced(n - 1, depth + 1) + fib_traced(n - 2, depth + 1)

print(fib_traced(4, 0))

```
fib(4)
    fib(3)
        fib(2)
            fib(1)
            fib(0)
        fib(1)
    fib(2)
        fib(1)
        fib(0)
3
```

Count them: 9 calls for `fib(4)`. And look at the repeats:

| call | how many times it was worked out |
| --- | --- |
| `fib(3)` | 1 |
| `fib(2)` | **2** |
| `fib(1)` | **3** |
| `fib(0)` | **2** |

It works out the same small answers again and again. For `fib(4)` that doesn't matter. For bigger numbers it gets very slow. You'll fix that later in this notebook with `@cache`.

**Try it:** run `fib_traced(5, 0)` and count the lines. It's 15 calls already.

---

## A helper function inside a LeetCode method

On LeetCode, your code lives inside `class Solution`. The usual way to write recursion there is to put a small **helper** function **inside** the method, and call it once to start things off.

A made-up problem: return the sum of `nums`. Here's the recursive way, with a helper that adds up from index `i` to the end:

In [ ]:
class Solution:
    def sumList(self, nums: list[int]) -> int:
        def add_from(i):
            if i == len(nums):
                return 0
            return nums[i] + add_from(i + 1)

        return add_from(0)

print(Solution().sumList([3, 1, 4]))

```
8
```

Things to notice:

- `def add_from(i):` is written **inside** `sumList`, indented one more level. It only exists inside `sumList`.
- The helper uses `nums` without it being passed in. A function written inside another one can **read** the outer function's variables.
- The helper carries an **index** `i` instead of slicing the list. The list stays the same and `i` moves along. That's the usual style, and it's faster than making a new list every call.
- The helper doesn't need `self`. Only the method has `self`.
- `return add_from(0)` starts the recursion at index 0 and returns its answer.

| call | `nums[i]` | works out | returns |
| --- | --- | --- | --- |
| `add_from(0)` | 3 | 3 + `add_from(1)` | 3 + 5 = **8** |
| `add_from(1)` | 1 | 1 + `add_from(2)` | 1 + 4 = 5 |
| `add_from(2)` | 4 | 4 + `add_from(3)` | 4 + 0 = 4 |
| `add_from(3)` | | `i == len(nums)`: base case | 0 |

In other people's solutions this helper is very often called **`dfs`**, short for **depth-first search**: "go as deep as you can before coming back", which is what these calls do. The name doesn't matter to Python. `dfs`, `helper`, `add_from` all work the same.

### Changing an outer variable: `nonlocal`

Now count the even numbers, with a counter `count` that the helper adds to. Run this; it breaks:

In [ ]:
class Solution:
    def countEvens(self, nums: list[int]) -> int:
        count = 0

        def visit(i):
            if i == len(nums):
                return
            if nums[i] % 2 == 0:
                count += 1
            visit(i + 1)

        visit(0)
        return count

print(Solution().countEvens([4, 7, 2, 8]))

```
UnboundLocalError: cannot access local variable 'count' where it is not associated with a value
```

Reading an outer variable is fine. **Changing** one is different. When a function puts a new value in a name (`count += 1` means `count = count + 1`), Python decides `count` is the helper's own new variable, a **local** one that belongs only to the helper. That local `count` has no value yet, so `count + 1` fails.

**`nonlocal count`** at the top of the helper says: "`count` isn't mine; I mean the one in the function outside".

In [ ]:
class Solution:
    def countEvens(self, nums: list[int]) -> int:
        count = 0

        def visit(i):
            nonlocal count
            if i == len(nums):
                return
            if nums[i] % 2 == 0:
                count += 1
            visit(i + 1)

        visit(0)
        return count

print(Solution().countEvens([4, 7, 2, 8]))

```
3
```

Rule of thumb: if the helper does `name = ...` or `name += ...` to an outer variable, it needs `nonlocal name`.

---

## Two styles: return the answer up, or collect into a list

There are two common ways for recursion to produce its answer. You'll see both in solutions.

**Style 1: return the answer up.** Each call returns its part, and the caller adds it on. No outer variable, so no `nonlocal`:

In [ ]:
class Solution:
    def countEvens(self, nums: list[int]) -> int:
        def evens_from(i):
            if i == len(nums):
                return 0
            rest = evens_from(i + 1)
            if nums[i] % 2 == 0:
                return 1 + rest
            return rest

        return evens_from(0)

print(Solution().countEvens([4, 7, 2, 8]))

```
3
```

| call | `nums[i]` | even? | `rest` | returns |
| --- | --- | --- | --- | --- |
| `evens_from(0)` | 4 | yes | 2 | **3** |
| `evens_from(1)` | 7 | no | 2 | 2 |
| `evens_from(2)` | 2 | yes | 1 | 2 |
| `evens_from(3)` | 8 | yes | 0 | 1 |
| `evens_from(4)` | | base case | | 0 |

Read the table from the **bottom up**: that's the order the answers come back.

**Style 2: collect into a list.** Make an empty list outside the helper, and let the helper `append` to it:

In [ ]:
class Solution:
    def findEvens(self, nums: list[int]) -> list[int]:
        found = []

        def visit(i):
            if i == len(nums):
                return
            if nums[i] % 2 == 0:
                found.append(nums[i])
            visit(i + 1)

        visit(0)
        return found

print(Solution().findEvens([4, 7, 2, 8]))

```
[4, 2, 8]
```

No `nonlocal` here. `found.append(...)` changes the list itself; it never puts a new value in the name `found`. It's like the lists notebook: changing what's on the shelf is fine, swapping the whole shelf is not.

**Which one when?**

- **Style 1** when the answer is one value built from smaller answers: a sum, a count, a biggest.
- **Style 2** when you're gathering many things along the way.

---

## `@cache`: remembering answers

Back to Fibonacci. How many calls does it really make? Here's `fib` as a helper with a call counter (using `nonlocal`):

In [ ]:
def count_calls(n):
    calls = 0

    def fib(k):
        nonlocal calls
        calls += 1
        if k < 2:
            return k
        return fib(k - 1) + fib(k - 2)

    answer = fib(n)
    print(f"fib({n}) = {answer}, using {calls} calls")

count_calls(10)
count_calls(20)
count_calls(25)

```
fib(10) = 55, using 177 calls
fib(20) = 6765, using 21891 calls
fib(25) = 75025, using 242785 calls
```

Almost a quarter of a million calls for `fib(25)`, because of all the repeats you saw in the call tree.

The fix: write each answer down the first time, and look it up after that, like keeping a notebook of sums you've already done. Python has this ready-made. It's called **`cache`** and lives in the **`functools`** toolbox:

In [ ]:
from functools import cache

def count_calls(n):
    calls = 0

    @cache
    def fib(k):
        nonlocal calls
        calls += 1
        if k < 2:
            return k
        return fib(k - 1) + fib(k - 2)

    answer = fib(n)
    print(f"fib({n}) = {answer}, using {calls} calls")

count_calls(10)
count_calls(20)
count_calls(25)

```
fib(10) = 55, using 11 calls
fib(20) = 6765, using 21 calls
fib(25) = 75025, using 26 calls
```

| `n` | calls without `@cache` | calls with `@cache` |
| --- | --- | --- |
| 10 | 177 | 11 |
| 20 | 21891 | 21 |
| 25 | 242785 | 26 |

Same answers, a tiny fraction of the work.

The only change is the line **`@cache`** right above `def fib`. A line starting with `@` above a `def` is called a **decorator**: it adds an extra behaviour to the function. `@cache` gives the function a memory:

- The first time `fib(7)` is asked for, it works out the answer and saves it.
- Every time after that, it hands back the saved answer straight away, without running the function again.

It's safe because `fib(7)` always gives the same answer. On LeetCode, `from functools import cache` is allowed; put it at the top.

---

## How deep can it go?

Python allows about 1000 calls waiting on the pile at once:

In [ ]:
import sys

print(sys.getrecursionlimit())

```
1000
```

If a problem's input is so big that the recursion goes deeper than that (for example, a list of 5000 items handled one call per item), you can raise the limit with `sys.setrecursionlimit(10000)` at the top of your code. You'll rarely need it in easy problems.

---

## Common mistakes

### Forgetting to start the helper, or to return its answer

Run this. The helper is fine, but look at the last line of `sumList`:

In [ ]:
class Solution:
    def sumList(self, nums: list[int]) -> int:
        def add_from(i):
            if i == len(nums):
                return 0
            return nums[i] + add_from(i + 1)

        add_from(0)

print(Solution().sumList([3, 1, 4]))

```
None
```

`add_from(0)` works out 8, and then the answer is thrown away. `sumList` itself never returns anything, so it gives back `None`. It must be `return add_from(0)`.

### `@cache` with a list as input

`@cache` keeps its answers in something like a dictionary, with the inputs as the keys. A list can't be a dictionary key, because a list can change. Run this:

In [ ]:
from functools import cache

@cache
def total(nums):
    if len(nums) == 0:
        return 0
    return nums[0] + total(nums[1:])

print(total([3, 1, 4]))

```
TypeError: unhashable type: 'list'
```

"Unhashable" means "can't be used as a key". The fix is the helper style from above: keep the list outside, and give the cached helper only numbers, like an index `i`.

### Calling a `Solution` method by its bare name

Inside `class Solution`, a method calls itself as `self.fib(n - 1)`, not `fib(n - 1)`. A helper function inside the method is called by its plain name. You saw this in the last notebook.

---

# ✏️ Your turn

Use recursion in every task. For the `class Solution` ones, a helper function inside the method is usually the easiest way.

### Task 1: count a letter (style 1: return the answer up)

Return how many times the letter `ch` appears in `s`. Use a helper `count_from(i)` that returns the count from index `i` to the end.

<details><summary>Hint</summary>

Base case: `i == len(s)` gives 0. Otherwise, work out `count_from(i + 1)` and add 1 if `s[i] == ch`.
</details>

In [ ]:
class Solution:
    def countChar(self, s: str, ch: str) -> int:
        # your code here
        pass

In [ ]:
from checks import check
check(Solution().countChar("banana", "a"), 3)
check(Solution().countChar("tea", "z"), 0)
check(Solution().countChar("", "a"), 0)

### Task 2: find the big ones (style 2: collect into a list)

Return a list of the numbers in `nums` that are **bigger than** `limit`, in the order they appear. Use a helper `visit(i)` that appends to a list made outside it.

In [ ]:
class Solution:
    def findBig(self, nums: list[int], limit: int) -> list[int]:
        # your code here
        pass

In [ ]:
check(Solution().findBig([5, 12, 3, 20], 10), [12, 20])
check(Solution().findBig([1, 2], 10), [])
check(Solution().findBig([], 10), [])

### Task 3: add up the even numbers (with `nonlocal`)

Return the sum of the even numbers in `nums`. This time, keep a `total = 0` in the method, and let a helper `visit(i)` add to it.

In [ ]:
class Solution:
    def sumEvens(self, nums: list[int]) -> int:
        # your code here
        pass

In [ ]:
check(Solution().sumEvens([4, 7, 2, 8]), 14)
check(Solution().sumEvens([1, 3]), 0)
check(Solution().sumEvens([]), 0)

### Task 4: LeetCode 509, Fibonacci Number

Return `fib(n)`: `fib(0) = 0`, `fib(1) = 1`, and every other one is the sum of the two before it.

In [ ]:
class Solution:
    def fib(self, n: int) -> int:
        # your code here
        pass

In [ ]:
check(Solution().fib(0), 0)
check(Solution().fib(1), 1)
check(Solution().fib(2), 1)
check(Solution().fib(4), 3)
check(Solution().fib(10), 55)

### Task 5: LeetCode 70, Climbing Stairs

There are `n` stairs. Each move you climb **1 or 2** stairs. In how many different ways can you reach the top?

For 3 stairs there are 3 ways: 1+1+1, 1+2, 2+1.

Think about the **last** move. To stand on stair `n`, you came either from stair `n - 1` (a 1-step) or from stair `n - 2` (a 2-step). So:

```
ways(n) = ways(n - 1) + ways(n - 2)
```

| `n` | 1 | 2 | 3 | 4 | 5 |
| --- | --- | --- | --- | --- | --- |
| ways | 1 | 2 | 3 | 5 | 8 |

The last check uses `n = 45`. Without `@cache` that takes billions of calls and never finishes. If the check cell keeps running (it shows `[*]` and no number), press the **■ stop** button in the toolbar, then add `@cache` to your helper.

<details><summary>Hint</summary>

A helper `ways(k)` inside the method, with `@cache` above it. Base cases: `ways(1)` is 1 and `ways(2)` is 2. Put `from functools import cache` at the top of the cell.
</details>

In [ ]:
class Solution:
    def climbStairs(self, n: int) -> int:
        # your code here
        pass

In [ ]:
check(Solution().climbStairs(1), 1)
check(Solution().climbStairs(2), 2)
check(Solution().climbStairs(3), 3)
check(Solution().climbStairs(5), 8)
check(Solution().climbStairs(45), 1836311903)

### Task 6: LeetCode 1137, N-th Tribonacci Number

Like Fibonacci, but each number is the sum of the **three** before it. `t(0) = 0`, `t(1) = 1`, `t(2) = 1`, and `t(n) = t(n - 1) + t(n - 2) + t(n - 3)`.

| `n` | 0 | 1 | 2 | 3 | 4 | 5 |
| --- | --- | --- | --- | --- | --- | --- |
| `t(n)` | 0 | 1 | 1 | 2 | 4 | 7 |

You'll need `@cache` again for `n = 25`.

In [ ]:
class Solution:
    def tribonacci(self, n: int) -> int:
        # your code here
        pass

In [ ]:
check(Solution().tribonacci(0), 0)
check(Solution().tribonacci(2), 1)
check(Solution().tribonacci(4), 4)
check(Solution().tribonacci(25), 1389537)

### Task 7: LeetCode 258, Add Digits

Add up the digits of `num`. If the answer has more than one digit, add up **its** digits, and keep going until one digit is left. Return that digit.

`38` → `3 + 8 = 11` → `1 + 1 = 2`. So the answer is 2.

<details><summary>Hint</summary>

Base case: a number below 10 is already one digit. Otherwise, work out the digit sum (with `% 10` and `// 10`), and call `self.addDigits` on that.
</details>

In [ ]:
class Solution:
    def addDigits(self, num: int) -> int:
        # your code here
        pass

In [ ]:
check(Solution().addDigits(38), 2)
check(Solution().addDigits(0), 0)
check(Solution().addDigits(9), 9)
check(Solution().addDigits(199), 1)

---

# ✅ Quiz: check yourself

Put your letter between the quotes, then run the check cell under it.

**Question 1.** Why does `count += 1` inside a helper need `nonlocal count`, but `found.append(x)` doesn't need `nonlocal found`?

- A) `+=` puts a new value in the name; `append` only changes the list that's already there
- B) Lists are faster than numbers

In [ ]:
answer_1 = "?"

In [ ]:
from checks import check_answer
check_answer("18-q1", answer_1)

**Question 2.** What does `@cache` do?

- A) It lets the recursion go deeper than 1000 calls
- B) It remembers each answer, so the same call isn't worked out twice

In [ ]:
answer_2 = "?"

In [ ]:
check_answer("18-q2", answer_2)

**Question 3.** Inside `class Solution`, the method `fib` wants to call itself. What do you write?

- A) `fib(n - 1)`
- B) `Solution(n - 1)`
- C) `self.fib(n - 1)`

In [ ]:
answer_3 = "?"

In [ ]:
check_answer("18-q3", answer_3)

**Question 4.** You want every word longer than 5 letters from a list. Which style fits best?

- A) Collect into a list
- B) Return one answer up

In [ ]:
answer_4 = "?"

In [ ]:
check_answer("18-q4", answer_4)

---

## Done

You can now follow recursion that calls itself twice, write a helper function inside a LeetCode method, change an outer variable with `nonlocal`, pick between returning the answer up and collecting into a list, and make slow recursion fast with `@cache`.

Stuck on a task? The answers are in `solutions/18-recursion-in-practice.ipynb`. Try for real first.

**Next:** writing your own classes, and "design" problems (`19-classes-and-design`).